# LungXplain v5 — baseline black-box cho RQ2

Notebook private này huấn luyện **DenseNet-121 dự đoán trực tiếp tử vong nội viện từ ảnh**
(không qua concept) và hai mô hình **late fusion** ghép điểm ảnh với 22 giá trị clinical
và 22 cờ thiếu. Mục đích: đo chi phí của nút cổ chai concept (RQ2) trên đúng cohort,
split và seed của ACC-ECBM.

Giao thức CNN giống hệt notebook ACC-ECBM native: 3 fold ngoài mẫu trên train, 10% nội bộ
để chọn checkpoint, tối đa 60 epoch, tối thiểu 10, patience 10, cùng optimizer, augment
và seed. Khác biệt duy nhất: nhãn là tử vong; checkpoint chọn bằng AUROC tử vong nội bộ.

Late fusion học trên **logit ngoài fold** của CNN cho train, nên không rò rỉ.
**Test không được dự đoán trong notebook này.** Dữ liệu và output phải giữ private.

## 1. Thư viện và cấu hình

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    interpret_version = importlib.metadata.version("interpret-core")
except importlib.metadata.PackageNotFoundError:
    interpret_version = None
if interpret_version != "0.7.8":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "interpret-core==0.7.8"])

import gc
import hashlib
import json
import math
import os
import random
import tarfile
import time
import types
import zipfile
from pathlib import Path, PurePosixPath

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             brier_score_loss, log_loss)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from interpret.glassbox import ExplainableBoostingClassifier

CFG = {
    "protocol": "lungxplain-v5-blackbox-1",
    "seeds": [42, 123, 2026],
    "outer_folds": 3, "inner_fraction": 0.10,
    "max_epochs": 60, "min_epochs": 10, "patience": 10,
    "batch_size": 16, "num_workers": 2, "image_size": 224,
    "backbone_lr": 1e-4, "head_lr": 3e-4, "weight_decay": 1e-5,
    "gradient_clip": 5.0, "pos_weight_cap": 10.0,
    "logistic_c": np.logspace(-4, 2, 13).tolist(), "logistic_cv_folds": 5,
    "ebm": {"interactions": 10, "max_rounds": 1500, "outer_bags": 4,
            "max_bins": 64, "max_interaction_bins": 16,
            "early_stopping_rounds": 50, "n_jobs": 4},
}
INPUT_ROOT = Path(os.environ.get("LX_INPUT_ROOT", "/kaggle/input"))
WORK = Path(os.environ.get("LX_WORK_ROOT", "/kaggle/working/lungxplain_blackbox"))
WORK.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def plain(value):
    if isinstance(value, dict):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(v) for v in value]
    if isinstance(value, np.ndarray):
        return plain(value.tolist())
    if isinstance(value, np.generic):
        return plain(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(json.dumps(plain(value), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)

ENVIRONMENT = {name: importlib.metadata.version(name) for name in
               ["torch", "torchvision", "numpy", "pandas", "scikit-learn", "interpret-core"]}
print({"device": str(DEVICE), "seeds": CFG["seeds"], "versions": ENVIRONMENT})

## 2. Nạp dữ liệu private và kiểm tra toàn vẹn

Cell này giống hệt notebook ACC-ECBM native: kiểm SHA-256 của bundle, manifest, schema,
validation report và từng ảnh cache. Nhãn tử vong của test giữ là NaN.

In [ ]:
EXPECTED_BUNDLE_SHA = "8b6d121c4f867c4598c012bb1f9fee43e1411247f946e0c1efd8cb7fcd143b4a"
EXPECTED_MANIFEST_SHA = "ac3a6119f4267b31b874c2198990d33593bd6ffbcd1aae4943733bddcc46899a"

def unpack_private_data(input_root, work):
    matches = sorted(input_root.rglob("bundle_manifest_v5.json"))
    matches = [p for p in matches if sha256(p) == EXPECTED_BUNDLE_SHA]
    if len(matches) != 1:
        raise RuntimeError("Attach the audited private v5 input dataset exactly once.")
    bundle_path = matches[0]
    bundle = json.loads(bundle_path.read_text())
    if bundle.get("format") != "lungxplain-private-v5" or bundle.get("private_required") is not True:
        raise RuntimeError("Unexpected dataset bundle.")
    source = bundle_path.parent
    destination = work / "input"
    destination.mkdir(parents=True, exist_ok=True)
    required = {name: digest for name, digest in bundle["members"].items() if name.startswith("data/")}
    aliases = bundle.get("aliases", {})
    for name in required:
        parts = PurePosixPath(name)
        if parts.is_absolute() or ".." in parts.parts or "\\" in name:
            raise RuntimeError("Unsafe data member path.")
    # A re-run reuses verified files; it never relies on a partial extraction.
    missing = [name for name, digest in required.items()
               if not (destination / name).is_file() or sha256(destination / name) != digest]
    archive_candidates = list(source.rglob("lungxplain_v5_data.tar"))
    if missing and archive_candidates:
        if len(archive_candidates) != 1 or sha256(archive_candidates[0]) != bundle["data_tar_sha256"]:
            raise RuntimeError("Data archive checksum mismatch.")
        storage_names = {aliases.get(name, name): name for name in required}
        with tarfile.open(archive_candidates[0], "r:") as archive:
            seen = set()
            for member in archive:
                storage_name = member.name.removeprefix("./")
                canonical = storage_names.get(storage_name)
                if member.isdir():
                    continue
                if canonical is None or not member.isfile() or canonical in seen:
                    raise RuntimeError("Unexpected or duplicate data archive member.")
                seen.add(canonical)
                target = destination / canonical
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.extractfile(member) as src, target.open("wb") as dst:
                    while block := src.read(8 * 1024 * 1024):
                        dst.write(block)
    elif missing:
        # Kaggle may already have expanded the outer archive.
        import shutil
        for name in missing:
            options = [source / aliases.get(name, name), source / name]
            options += list(source.glob("**/" + aliases.get(name, name)))
            candidate = next((p for p in options if p.is_file() and sha256(p) == required[name]), None)
            if candidate is None:
                raise RuntimeError("Missing or altered v5 data file.")
            target = destination / name
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(candidate, target)
    for name, digest in required.items():
        if not (destination / name).is_file() or sha256(destination / name) != digest:
            raise RuntimeError("Extracted data failed checksum verification.")
    print({"verified_data_files": len(required), "source_code_loaded": False})
    return destination, bundle

DATA_ROOT, BUNDLE = unpack_private_data(INPUT_ROOT, WORK)
DATASET_DIR = DATA_ROOT / BUNDLE["dataset_dir"]
PIXEL_DIR = DATA_ROOT / BUNDLE["pixel_cache"]
MANIFEST = DATASET_DIR / "manifest.csv"
if sha256(MANIFEST) != EXPECTED_MANIFEST_SHA:
    raise RuntimeError("The manifest is not the audited v5 cohort.")
SCHEMA = json.loads((DATASET_DIR / "feature_schema.json").read_text())
VALIDATION = json.loads((DATASET_DIR / "validation_report.json").read_text())
exact = VALIDATION.get("exact_5000_validation", {})
if VALIDATION.get("passed") is not True or exact != {"valid_images": 5000, "required": 5000}:
    raise RuntimeError("A passing exact-5000 dataset validation is required.")
FRAME = pd.read_csv(MANIFEST, low_memory=False)
CONCEPTS = list(SCHEMA["concept_label_columns"])
CLINICAL = list(SCHEMA["model_clinical_columns"])
if len(FRAME) != 5000 or len(CONCEPTS) != 14 or len(CLINICAL) != 44:
    raise RuntimeError("Unexpected cohort dimensions.")
for column in ["subject_id", "hadm_id", "dicom_id"]:
    if FRAME[column].isna().any() or FRAME[column].nunique() != 5000:
        raise RuntimeError("Patient/admission/image uniqueness failed.")
if FRAME["split"].value_counts().to_dict() != {"train": 4000, "valid": 500, "test": 500}:
    raise RuntimeError("Unexpected patient split.")
CACHE_INDEX = json.loads((PIXEL_DIR / "index.json").read_text())
if (CACHE_INDEX["rows"] != 5000 or len(CACHE_INDEX["images"]) != 5000 or
        CACHE_INDEX["configuration"]["manifest_sha256"] != EXPECTED_MANIFEST_SHA):
    raise RuntimeError("Pixel cache row alignment failed.")
IMAGE_PATHS = [PIXEL_DIR / item["file"] for item in CACHE_INDEX["images"]]
for path, entry in zip(IMAGE_PATHS, CACHE_INDEX["images"]):
    if not path.is_file() or sha256(path) != entry["sha256"]:
        raise RuntimeError("Pixel cache integrity failed.")
TRAIN = np.flatnonzero(FRAME["split"].eq("train"))
VALID = np.flatnonzero(FRAME["split"].eq("valid"))
TEST = np.flatnonzero(FRAME["split"].eq("test"))
TARGETS = FRAME[CONCEPTS].to_numpy(dtype=np.float32)
SUPERVISED = (FRAME[["ignore_" + c for c in CONCEPTS]].to_numpy() == 0)
KNOWN = (FRAME[["known_" + c for c in CONCEPTS]].to_numpy() == 1)
if not np.isin(TARGETS, [0, 1]).all():
    raise RuntimeError("Concept targets must be binary.")
# Mortality labels are materialized only for train/validation. Test entries stay NaN.
OUTCOME = np.full(len(FRAME), np.nan)
development = np.r_[TRAIN, VALID]
OUTCOME[development] = FRAME.loc[development, SCHEMA["outcome_column"]].to_numpy(dtype=float)
del development

def clinical_matrix(frame):
    result = frame[CLINICAL].copy()
    for column in CLINICAL:
        if column in SCHEMA["categorical_value_columns"]:
            mapping = {value: index for index, value in enumerate(SCHEMA["categorical_values"][column])}
            if not frame[column].dropna().isin(mapping).all():
                raise ValueError("Unknown clinical category.")
            result[column] = frame[column].map(mapping)
        else:
            result[column] = pd.to_numeric(result[column], errors="raise")
    return result.to_numpy(dtype=np.float64)

Z = clinical_matrix(FRAME)
FEATURE_NAMES = CONCEPTS + CLINICAL
FEATURE_TYPES = ["continuous"] * len(CONCEPTS) + [
    "nominal" if c in SCHEMA["categorical_value_columns"] or c.endswith("_missing") else "continuous"
    for c in CLINICAL]
WEIGHTS_PATH = DATA_ROOT / "data/model_cache/torch/checkpoints/densenet121-a639ec97.pth"
torch.hub.set_dir(str(WEIGHTS_PATH.parent.parent))
print({"patients": len(FRAME), "split": FRAME["split"].value_counts().to_dict(),
       "concepts": len(CONCEPTS), "clinical_inputs": len(CLINICAL), "outcome_inputs": len(FEATURE_NAMES),
       "concept_training": "U-Ignore; unmentioned treated as negative",
       "intervention_references": "explicit positive/negative only", "test_evaluated": False})

## 3. Dataset và DenseNet-121 dự đoán tử vong

Dataset chỉ trả ảnh và chỉ số hàng; nhãn lấy từ `OUTCOME`, nên hàng test (NaN) không thể
lọt vào loss. `pos_weight` tính trên fitting rows của từng fold, chặn tối đa 10.

In [ ]:
class MortalityImageDataset(Dataset):
    def __init__(self, rows, training=False):
        self.rows = np.asarray(rows, dtype=int)
        steps = []
        if training:
            steps.append(transforms.RandomAffine(5, translate=(0.02, 0.02), scale=(0.98, 1.02)))
        steps += [transforms.CenterCrop(CFG["image_size"]), transforms.ToTensor(),
                  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, item):
        row = self.rows[item]
        with Image.open(IMAGE_PATHS[row]) as image:
            x = self.transform(image.convert("RGB"))
        return x, int(row)

def worker_seed(worker_id):
    value = torch.initial_seed() % 2**32
    np.random.seed(value)
    random.seed(value)

def make_loader(rows, training=False, seed=0):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(MortalityImageDataset(rows, training), batch_size=CFG["batch_size"],
                      shuffle=training, num_workers=CFG["num_workers"],
                      pin_memory=AMP, generator=generator, worker_init_fn=worker_seed)

def build_mortality_cnn():
    model = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
    model.classifier = nn.Linear(model.classifier.in_features, 1)
    for parameter in model.parameters():
        parameter.requires_grad_(True)
    return model.to(DEVICE)

def mortality_labels(rows):
    labels = OUTCOME[np.asarray(rows, dtype=int)]
    if not np.isfinite(labels).all():
        raise RuntimeError("Mortality labels requested outside train/validation.")
    return labels

def mortality_pos_weight(rows):
    labels = mortality_labels(rows)
    weight = np.clip((labels == 0).sum() / max((labels == 1).sum(), 1), 1, CFG["pos_weight_cap"])
    return torch.tensor([weight], dtype=torch.float32, device=DEVICE)

def predict_mortality(model, rows):
    model.eval()
    batches = []
    with torch.inference_mode():
        for x, _ in make_loader(rows):
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                probability = torch.sigmoid(model(x.to(DEVICE, non_blocking=True))[:, 0])
            batches.append(probability.float().cpu().numpy())
    return np.concatenate(batches).astype(np.float64)

def probability_logit(probabilities):
    p = np.clip(np.asarray(probabilities, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

## 4. Vòng lặp huấn luyện, cross-fitting và refit

Mỗi fold huấn luyện trên hai fold còn lại (trừ 10% nội bộ để chọn checkpoint) và dự đoán
fold giữ lại. Refit cuối trên toàn bộ train với số epoch là trung vị epoch được chọn,
rồi dự đoán validation. Checkpoint lưu sau mỗi epoch để có thể tiếp tục.

In [ ]:
def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    torch.save(value, temporary)
    temporary.replace(path)

def train_stage(folder, fitting_rows, selection_rows, stage_seed, refit_epochs=None):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    fitting_rows = np.asarray(fitting_rows, dtype=int)
    selection_rows = np.asarray(selection_rows, dtype=int)
    if np.intersect1d(fitting_rows, selection_rows).size:
        raise ValueError("Fitting and selection patients overlap.")
    if not np.isin(np.r_[fitting_rows, selection_rows], TRAIN).all():
        raise ValueError("CNN fitting/selection must stay inside train.")
    stage_spec = {"fitting_rows": fitting_rows.tolist(), "selection_rows": selection_rows.tolist(),
                  "stage_seed": stage_seed, "refit_epochs": refit_epochs}
    spec_path = folder / "stage.json"
    if spec_path.exists() and json.loads(spec_path.read_text()) != stage_spec:
        raise RuntimeError("Stage split/configuration changed. Use a new run directory.")
    save_json(spec_path, stage_spec)
    seed_all(stage_seed)
    model = build_mortality_cnn()
    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": CFG["backbone_lr"]},
        {"params": model.classifier.parameters(), "lr": CFG["head_lr"]}],
        weight_decay=CFG["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    pos_weight = mortality_pos_weight(fitting_rows)
    last_path, best_path = folder / "last.pt", folder / "best.pt"
    history, start, best, best_epoch, stale, steps = [], 0, -math.inf, 0, 0, 0
    done = False
    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        optimizer.load_state_dict(checkpoint["optimizer"])
        scaler.load_state_dict(checkpoint["scaler"])
        history = checkpoint["history"]
        start, best, best_epoch = checkpoint["epoch"], checkpoint["best"], checkpoint["best_epoch"]
        stale, steps, done = checkpoint["stale"], checkpoint["steps"], checkpoint["done"]
        del checkpoint
    limit = int(refit_epochs if refit_epochs is not None else CFG["max_epochs"])
    for epoch in range(start + 1, limit + 1) if not done else []:
        epoch_seed = stage_seed * 1000 + epoch
        seed_all(epoch_seed)
        started = time.monotonic()
        model.train()
        total_loss, total_rows = 0.0, 0
        for x, rows in make_loader(fitting_rows, training=True, seed=epoch_seed):
            labels = torch.as_tensor(mortality_labels(rows.numpy()), dtype=torch.float32, device=DEVICE)
            x = x.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                logits = model(x)[:, 0]
            loss = F.binary_cross_entropy_with_logits(logits.float(), labels, pos_weight=pos_weight)
            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite CNN loss.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["gradient_clip"])
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            steps += int(scaler.get_scale() >= old_scale)
            total_loss += float(loss.detach()) * len(labels)
            total_rows += len(labels)
        score = None
        improved = False
        if refit_epochs is None:
            p_inner = predict_mortality(model, selection_rows)
            score = float(roc_auc_score(mortality_labels(selection_rows), p_inner))
            improved = score > best + 1e-8
            if improved:
                best, best_epoch, stale = score, epoch, 0
            else:
                stale += 1
        else:
            best_epoch = epoch
        done = epoch == limit or (refit_epochs is None and epoch >= CFG["min_epochs"] and stale >= CFG["patience"])
        record = {"epoch": epoch, "mortality_loss": total_loss / max(1, total_rows),
                  "inner_mortality_auroc": score, "seconds": time.monotonic() - started,
                  "optimizer_steps": steps}
        history.append(record)
        checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                      "scaler": scaler.state_dict(), "epoch": epoch, "best": best,
                      "best_epoch": best_epoch, "stale": stale, "steps": steps,
                      "done": done, "history": history, "pos_weight": pos_weight.cpu()}
        if improved or refit_epochs is not None:
            atomic_torch_save({"model": model.state_dict(), "epoch": epoch}, best_path)
        atomic_torch_save(checkpoint, last_path)
        save_json(folder / "history.json", history)
        print({"stage": folder.name, **plain(record)}, flush=True)
        del checkpoint
    selected = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(selected["model"])
    del selected, optimizer, scaler
    return model, {"best_epoch": best_epoch, "epochs": len(history), "optimizer_steps": steps}

def crossfit_blackbox(seed, run):
    cache = run / "image_mortality_predictions.npz"
    if cache.exists():
        with np.load(cache) as saved:
            result = saved["probabilities"].copy()
            if (not np.array_equal(saved["train_rows"], TRAIN) or
                    not np.array_equal(saved["valid_rows"], VALID) or
                    result.shape != (len(FRAME),) or
                    not np.isfinite(result[np.r_[TRAIN, VALID]]).all() or
                    not np.isnan(result[TEST]).all() or (saved["fold_id"][TRAIN] < 0).any()):
                raise RuntimeError("Cached prediction alignment or test isolation failed.")
            return result
    probabilities = np.full(len(FRAME), np.nan, dtype=np.float64)
    fold_id = np.full(len(FRAME), -1, dtype=int)
    selected_epochs, stages = [], []
    outer = StratifiedKFold(n_splits=CFG["outer_folds"], shuffle=True, random_state=seed)
    for fold, (development_positions, heldout_positions) in enumerate(outer.split(TRAIN, OUTCOME[TRAIN])):
        development_rows, heldout_rows = TRAIN[development_positions], TRAIN[heldout_positions]
        fitting_rows, inner_rows = train_test_split(
            development_rows, test_size=CFG["inner_fraction"], random_state=seed + fold,
            stratify=OUTCOME[development_rows])
        if np.intersect1d(heldout_rows, np.r_[fitting_rows, inner_rows]).size:
            raise RuntimeError("Outer fold leakage.")
        model, info = train_stage(run / f"fold{fold}", fitting_rows, inner_rows, seed + fold)
        probabilities[heldout_rows] = predict_mortality(model, heldout_rows)
        fold_id[heldout_rows] = fold
        selected_epochs.append(info["best_epoch"])
        stages.append(info)
        del model
        gc.collect()
        if AMP:
            torch.cuda.empty_cache()
    refit_epochs = max(1, int(np.median(selected_epochs)))
    model, info = train_stage(run / "final_cnn", TRAIN, [], seed + 100, refit_epochs=refit_epochs)
    # Test predictions are not computed here; their entries remain NaN.
    probabilities[VALID] = predict_mortality(model, VALID)
    del model
    if not np.isfinite(probabilities[np.r_[TRAIN, VALID]]).all() or np.any(fold_id[TRAIN] < 0):
        raise RuntimeError("Incomplete out-of-fold predictions.")
    temporary = cache.with_name(cache.name + ".partial")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, probabilities=probabilities, fold_id=fold_id,
                            train_rows=TRAIN, valid_rows=VALID)
    temporary.replace(cache)
    save_json(run / "cnn_summary.json", {"folds": stages, "final": info,
                                         "refit_epochs": refit_epochs, "test_predicted": False})
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()
    return probabilities

## 5. Late fusion và metric

Ba baseline mỗi seed:

- `image_only_cnn`: xác suất của CNN refit trên validation.
- `late_fusion_ebm`: EBM (cùng hyperparameter với ACC-ECBM) trên logit ảnh ngoài fold
  + 22 clinical + 22 cờ thiếu.
- `late_fusion_logistic`: logistic trên cùng 45 đầu vào; imputation/scale nằm trong
  pipeline, C chọn bằng CV 5 fold log-loss trên train.

Ngưỡng mỗi mô hình chọn bằng Macro-F1 trên validation.

In [ ]:
FUSION_NAMES = ["image_mortality_logit"] + CLINICAL
FUSION_TYPES = ["continuous"] + [
    "nominal" if c in SCHEMA["categorical_value_columns"] or c.endswith("_missing") else "continuous"
    for c in CLINICAL]

def outcome_metrics(labels, probabilities, threshold):
    return {"auroc": roc_auc_score(labels, probabilities),
            "auprc": average_precision_score(labels, probabilities),
            "macro_f1": f1_score(labels, probabilities >= threshold, average="macro", zero_division=0),
            "brier": brier_score_loss(labels, probabilities),
            "logloss": log_loss(labels, probabilities, labels=[0, 1])}

def select_threshold(labels, probabilities):
    candidates = np.unique(np.r_[0.0, probabilities, np.nextafter(1.0, 2.0)])
    scores = [f1_score(labels, probabilities >= t, average="macro", zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])

def fusion_inputs(image_probabilities, rows):
    return np.column_stack([probability_logit(image_probabilities[rows]), Z[rows]])

def fit_fusion_ebm(image_probabilities, seed):
    inputs = fusion_inputs(image_probabilities, TRAIN)
    if inputs.shape[1] != 45:
        raise ValueError("Late fusion must receive one image score + 44 clinical columns.")
    ebm = ExplainableBoostingClassifier(feature_names=FUSION_NAMES, feature_types=FUSION_TYPES,
                                      random_state=seed, **CFG["ebm"])
    return ebm.fit(inputs, mortality_labels(TRAIN).astype(int))

def make_logistic(names, seed):
    categorical = [i for i, name in enumerate(names) if name == "clinical_sex_recorded"]
    numeric = [i for i in range(len(names)) if i not in categorical]
    blocks = [("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler())]), numeric)]
    if categorical:
        blocks.append(("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical))
    pipeline = Pipeline([("preprocess", ColumnTransformer(blocks)),
                         ("logistic", LogisticRegression(max_iter=5000, random_state=seed))])
    return GridSearchCV(pipeline, {"logistic__C": CFG["logistic_c"]},
                        cv=StratifiedKFold(CFG["logistic_cv_folds"], shuffle=True, random_state=seed),
                        scoring="neg_log_loss", refit=True, n_jobs=1, error_score="raise")

def fit_fusion_logistic(image_probabilities, seed):
    search = make_logistic(FUSION_NAMES, seed)
    return search.fit(fusion_inputs(image_probabilities, TRAIN), mortality_labels(TRAIN).astype(int))

## 6. Chạy từng seed và lưu kết quả

Run gắn với hash dữ liệu, code, cấu hình và môi trường (cùng cơ chế với notebook ACC-ECBM).
Mỗi seed lưu checkpoint CNN, dự đoán ngoài fold, mô hình fusion, ngưỡng, dự đoán validation
và một ZIP. Không có dự đoán test.

In [ ]:
def stable_code(function):
    def encode(value):
        if isinstance(value, types.CodeType):
            return {"bytecode": value.co_code.hex(), "constants": [encode(c) for c in value.co_consts],
                    "names": value.co_names, "variables": value.co_varnames,
                    "freevars": value.co_freevars, "cellvars": value.co_cellvars,
                    "argcount": value.co_argcount, "kwonlyargcount": value.co_kwonlyargcount}
        if isinstance(value, (tuple, list)):
            return [encode(v) for v in value]
        if isinstance(value, (str, int, float, bool)) or value is None:
            return value
        return repr(value)
    return {"body": encode(function.__code__), "defaults": repr(function.__defaults__),
            "kwdefaults": repr(function.__kwdefaults__)}

def implementation_fingerprint():
    definitions = {}
    for name, value in list(globals().items()):
        if isinstance(value, types.FunctionType) and value.__module__ == __name__:
            definitions[name] = stable_code(value)
        elif isinstance(value, type) and value.__module__ == __name__:
            definitions[name] = {method: stable_code(fn) for method, fn in vars(value).items()
                                 if isinstance(fn, types.FunctionType)}
    return hashlib.sha256(json.dumps(definitions, sort_keys=True).encode()).hexdigest()

def bind_run(run, seed):
    binding = {"configuration": CFG, "seed": seed, "code_sha256": implementation_fingerprint(),
               "manifest_sha256": sha256(MANIFEST),
               "schema_sha256": sha256(DATASET_DIR / "feature_schema.json"),
               "cache_index_sha256": sha256(PIXEL_DIR / "index.json"),
               "imagenet_sha256": sha256(WEIGHTS_PATH), "environment": ENVIRONMENT,
               "device": str(DEVICE), "test_evaluated": False}
    run.mkdir(parents=True, exist_ok=True)
    binding_file = run / "binding.json"
    if binding_file.exists() and json.loads(binding_file.read_text()) != binding:
        raise RuntimeError("Run binding changed. Set a new WORK directory before training.")
    save_json(binding_file, binding)

def run_seed(seed):
    run = WORK / "runs" / f"seed{seed}"
    bind_run(run, seed)
    complete = run / "complete.json"
    if complete.exists():
        stamp = json.loads(complete.read_text())
        for name, digest in stamp["artifacts"].items():
            if not (run / name).is_file() or sha256(run / name) != digest:
                raise RuntimeError("Completed run artifact changed.")
        return json.loads((run / "validation_summary.json").read_text())
    seed_all(seed)
    image = crossfit_blackbox(seed, run)
    y_valid = mortality_labels(VALID).astype(int)
    predictions = {"image_only_cnn": image[VALID]}
    ebm = fit_fusion_ebm(image, seed)
    joblib.dump(ebm, run / "late_fusion_ebm.joblib")
    predictions["late_fusion_ebm"] = ebm.predict_proba(fusion_inputs(image, VALID))[:, 1]
    save_json(run / "late_fusion_ebm_global_explanation.json", ebm.explain_global().data())
    logistic = fit_fusion_logistic(image, seed)
    joblib.dump(logistic.best_estimator_, run / "late_fusion_logistic.joblib")
    predictions["late_fusion_logistic"] = logistic.predict_proba(fusion_inputs(image, VALID))[:, 1]
    save_json(run / "late_fusion_logistic_selection.json",
              {"best_c": logistic.best_params_["logistic__C"], "cv_neg_logloss": logistic.best_score_,
               "candidates": CFG["logistic_c"], "selection_split": "train_cv"})
    summaries = []
    for variant, prediction in predictions.items():
        threshold = select_threshold(y_valid, prediction)
        metrics = outcome_metrics(y_valid, prediction, threshold)
        summaries.append({"seed": seed, "variant": variant, "threshold": threshold, **metrics})
        print({"seed": seed, "variant": variant, **plain(metrics)}, flush=True)
    np.savez_compressed(run / "outcome_validation.npz", rows=VALID,
                        **{name: values for name, values in predictions.items()})
    save_json(run / "validation_summary.json", summaries)
    artifacts = {str(path.relative_to(run)): sha256(path) for path in sorted(run.rglob("*"))
                 if path.is_file() and path.name != "complete.json" and not path.name.endswith(".partial")}
    save_json(complete, {"seed": seed, "artifacts": artifacts, "test_evaluated": False})
    archive_path = WORK / f"lungxplain_blackbox_seed{seed}.zip"
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(run.rglob("*")):
            if path.is_file() and not path.name.endswith(".partial"):
                archive.write(path, arcname=str(path.relative_to(run.parent)))
    return summaries

In [ ]:
if DEVICE.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU before running the full training cell.")
ALL_RESULTS = []
for seed in CFG["seeds"]:
    print({"starting_seed": seed, "test_evaluated": False}, flush=True)
    ALL_RESULTS.extend(run_seed(seed))
RESULT_TABLE = pd.DataFrame(ALL_RESULTS)
RESULT_TABLE.to_csv(WORK / "validation_all_seeds.csv", index=False)
display(RESULT_TABLE)

## 7. Tổng hợp validation

Mean ± SD chỉ tính khi có nhiều seed. So sánh có ghép cặp với ACC-ECBM được làm ở bước
phân tích, trên cùng bệnh nhân validation.

In [ ]:
metrics = ["auroc", "auprc", "macro_f1", "brier", "logloss"]
seed_count = RESULT_TABLE.groupby("variant")["seed"].nunique()
if (seed_count > 1).all():
    display(RESULT_TABLE.groupby("variant")[metrics].agg(["mean", "std"]))
else:
    print("Kết quả một seed; không báo cáo mean ± SD.")
save_json(WORK / "experiment_status.json", {
    "dataset": "v5", "completed_seeds": sorted(RESULT_TABLE.seed.unique().tolist()),
    "evaluation_split": "valid", "test_evaluated": False,
    "source_code_loaded_from_dataset": False, "configuration": CFG})
print("Đã lưu kết quả validation và checkpoint tại:", WORK)